## Data

### 1. Load the pinned snapshot

In [3]:
import json
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT.parent if ROOT.name == "research" else ROOT
config = json.loads((ROOT / "research" / "source_config.json").read_text())
snapshot = ROOT / "data" / "raw" / "statsbomb" / config["source_ref"]
catalog_path = (
    snapshot
    / "matches"
    / str(config["competition_id"])
    / f"{config['season_id']}.json"
)
matches = json.loads(catalog_path.read_text())
SHOOTOUT_PERIOD = 5

print("Source scope")
print(json.dumps({
    "source_ref": config["source_ref"],
    "competition_id": config["competition_id"],
    "season_id": config["season_id"],
    "catalog_path": str(catalog_path.relative_to(ROOT)),
    "matches_observed": len(matches),
}, indent=2))

Source scope
{
  "source_ref": "b0bc9f22dd77c206ddedc1d742893b3bbe64baec",
  "competition_id": 55,
  "season_id": 282,
  "catalog_path": "data/raw/statsbomb/b0bc9f22dd77c206ddedc1d742893b3bbe64baec/matches/55/282.json",
  "matches_observed": 51
}


### 2. Inspect match fields and participant keys

In [4]:
def field_profile(field_counts, record_count):
    return {
        "records": record_count,
        "observed_on_every_record": [
            field
            for field, count in sorted(field_counts.items())
            if count == record_count
        ],
        "optional_or_type_specific": {
            field: count
            for field, count in sorted(field_counts.items())
            if count < record_count
        },
    }


match_field_counts = {}
for match in matches:
    for field in match:
        match_field_counts[field] = match_field_counts.get(field, 0) +1

match_ids = [match.get("match_id") for match in matches]
match_ids_present = [
    match_id
    for match_id in match_ids
    if match_id is not None
]
missing_match_ids = len(match_ids) - len(match_ids_present)
duplicate_match_ids = (
    len(match_ids_present) - len(set(match_ids_present))
)

participants = {}
invalid_match_participants = 0
for match in matches:
    match_id = match.get("match_id")
    team_ids = {
        (match.get("home_team") or {}).get("home_team_id"),
        (match.get("away_team") or {}).get("away_team_id"),
    }
    if match_id is not None:
        participants[match_id] = team_ids
    if len(team_ids) != 2 or None in team_ids:
        invalid_match_participants += 1

match_fields = field_profile(match_field_counts, len(matches))
match_checks = {
    "missing_match_ids": missing_match_ids,
    "duplicate_match_ids": duplicate_match_ids,
    "invalid_match_participants": invalid_match_participants,
}

print("Match catalog scan")
print(json.dumps({
    "fields": match_fields,
    "checks": match_checks,
}, indent=2))

Match catalog scan
{
  "fields": {
    "records": 51,
    "observed_on_every_record": [
      "away_score",
      "away_team",
      "competition",
      "competition_stage",
      "home_score",
      "home_team",
      "kick_off",
      "last_updated",
      "last_updated_360",
      "match_date",
      "match_id",
      "match_status",
      "match_status_360",
      "match_week",
      "metadata",
      "referee",
      "season",
      "stadium"
    ],
    "optional_or_type_specific": {}
  },
  "checks": {
    "missing_match_ids": 0,
    "duplicate_match_ids": 0,
    "invalid_match_participants": 0
  }
}
